To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Introducing **Unsloth Studio** - a new open source, no-code web UI to train and run LLMs. [Blog](https://unsloth.ai/docs/new/studio) • [Notebook](https://colab.research.google.com/github/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb)

<table><tr>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FxV1PO5DbF3ksB51nE2Tw%252Fmore%2520cropped%2520ui%2520for%2520homepage.png%3Falt%3Dmedia%26token%3Df75942c9-3d8d-4b59-8ba2-1a4a38de1b86&width=376&dpr=3&quality=100&sign=a663c397&sv=2" width="200" height="120" alt="Unsloth Studio Training UI"></a><br><sub><b>Train models</b> — no code needed</sub></td>
<td align="center"><a href="https://unsloth.ai/docs/new/studio"><img src="https://unsloth.ai/docs/~gitbook/image?url=https%3A%2F%2F3215535692-files.gitbook.io%2F~%2Ffiles%2Fv0%2Fb%2Fgitbook-x-prod.appspot.com%2Fo%2Fspaces%252FxhOjnexMCB3dmuQFQ2Zq%252Fuploads%252FRCnTAZ6Uh88DIlU3g0Ij%252Fmainpage%2520unsloth.png%3Falt%3Dmedia%26token%3D837c96b6-bd09-4e81-bc76-fa50421e9bfb&width=376&dpr=3&quality=100&sign=c1a39da1&sv=2" width="200" height="120" alt="Unsloth Studio Chat UI"></a><br><sub><b>Run GGUF models</b> on Mac, Windows & Linux</sub></td>
</tr></table>

Train MoEs - DeepSeek, GLM, Qwen and gpt-oss 12x faster with 35% less VRAM. [Blog](https://unsloth.ai/docs/new/faster-moe)

Ultra Long-Context Reinforcement Learning is here with 7x more context windows! [Blog](https://unsloth.ai/docs/new/grpo-long-context)

New in Reinforcement Learning: [FP8 RL](https://unsloth.ai/docs/new/fp8-reinforcement-learning) • [Vision RL](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl) • [Standby](https://unsloth.ai/docs/basics/memory-efficient-rl) • [gpt-oss RL](https://unsloth.ai/docs/new/gpt-oss-reinforcement-learning)

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/unsloth-model-catalog) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps tokenizers trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
# causal_conv1d is supported only on torch==2.8.0. If you have newer torch versions, please wait 10 minutes!
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0
!pip uninstall torchcodec -y

### Unsloth

In [1]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 1536 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # We also uploaded 4bit for 405b!
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    # Can select any from the below:
    # "unsloth/Qwen2.5-0.5B", "unsloth/Qwen2.5-1.5B", "unsloth/Qwen2.5-3B"
    # "unsloth/Qwen2.5-14B",  "unsloth/Qwen2.5-32B",  "unsloth/Qwen2.5-72B",
    # And also all Instruct versions and Math. Coding verisons!
    model_name = "unsloth/Qwen2.5-7B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = False,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

ModuleNotFoundError: No module named 'unsloth'

We now add LoRA adapters for parameter efficient finetuning - this allows us to only efficiently train 1% of all parameters.

**[NEW]** We also support finetuning ONLY the vision part of the model, or ONLY the language part. Or you can select both! You can also select to finetune the attention or the MLP layers!

Lora

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 8,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

In [ ]:
model = model.bfloat16()  # 用 bf16 节省显存

<a name="Data"></a>
### Data Prep
We'll be using a sampled dataset of handwritten maths formulas. The goal is to convert these images into a computer readable form - ie in LaTeX form, so we can render it. This can be very useful for complex formulas.

You can access the dataset [here](https://huggingface.co/datasets/unsloth/LaTeX_OCR). The full dataset is [here](https://huggingface.co/datasets/linxy/LaTeX_OCR).

In [ ]:
from datasets import load_dataset

# 替换为你自己的用户名和仓库名
repo_id = "Spiderman01/soulchat_split_raw"

# 一键加载整个 DatasetDict
dataset = load_dataset(repo_id)

# 查看结构
print(dataset)

# 访问特定的分片
train_data = dataset['train']
test_data = dataset['test']
human_data = dataset['human_validation']

print(f"训练集第一条对话的主题: {train_data[0]['topic']}")

In [ ]:
train_data[0]

{'id': 116193,
 'topic': '家庭',
 'messages': [{'content': '我生长在缺爱的家族里。父亲的兄弟姐妹间比较寡淡。也会互帮互助，但是总感觉没那么舒心和睦信任。隔着什么似的不亲。在只讲理不讲爱的家族里，大家战战兢兢如履薄冰慌乱，因为喜欢斗来斗去，凡事都要争个对错，大部分时间比较自私有时也会无私（自我保护?）',
   'role': 'user'},
  {'content': '孤独、不信任的环境让你倍感压力，这很正常。我可以先听你诉说，真正了解你的经历，帮助你和家族进行沟通和理解。',
   'role': 'assistant'},
  {'content': '父亲是长兄为父（爷爷过世的早）父亲不会表达爱，他的爱像是以爱之名的控制。他的兄弟姐妹像是靠着他（经济上）却又反感他的控制。小叔，姑姑找的对象更像是太博爱（水性杨花的渣男渣女）。我爸找的我妈则是一味的圣母情节（也是太博爱?）',
   'role': 'user'},
  {'content': '你的父母和亲戚之间的模式很常见，但并不健康。你可能会通过讨论和思考找到实现改变的方法，或者进一步明白为什么这种模式发展成现在的样子。',
   'role': 'assistant'},
  {'content': '唯独大叔找的婶婶是会爱懂爱的人，这个大家族就属大叔家比较和谐，因此孩子没有受到影响。从小缺爱，又一直被经济控制家庭暴力成长下的我，我缺爱也怕缺钱，因此既要另一半有钱有能力能保护我，又要他懂爱会爱，但是很难找，虽然我也很努力经济独立，精神也慢慢学会独立，但是总怕自己哪天不利，没有人支撑的住我，所以找对象一直挑剔，如何克服这种心理',
   'role': 'user'},
  {'content': '你来这里寻求帮助，说明你想要自我成长，这个态度很重要。无论你发现在哪些方面需要发展，我会一直帮助你，给你建议和支持，帮你缓解压力，一步一步走出寻求支撑的恐惧。',
   'role': 'assistant'},
  {'content': '我明白，但我感到自己在生活中无法正常地交朋友和恋爱，总有一些情绪和心理问题困扰着我，怎么办呢？',
   'role': 'user'},
  {'content': '这是很理解的。我们可以探讨一些可能的原因和

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Let's first see before we do any finetuning what the model outputs for the first example!

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer


user_query = """我生长在缺爱的家族里。父亲的兄弟姐妹间比较寡淡。也会互帮互助，但是总感觉没那么舒心和睦信任。隔着什么似的不亲。在只讲理不讲爱的家族里，大家战战兢兢如履薄冰慌乱，因为喜欢斗来斗去，凡事都要争个对错，大部分时间比较自私有时也会无私（自我保护?）"""

messages = [
    {"role": "system", "content": "你是一位资深的心理咨询师。请基于对话历史，用温暖、专业、共情的语气回复用户。"},
    {"role": "user", "content": user_query}
]



inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=1024)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))


感谢您分享这段经历，听起来您成长在一个对情感表达相对克制的家庭环境中。这样的家庭氛围确实可能让人感到疏离和不安，尤其是在情感需求得不到满足的时候。在这种环境下成长，可能会让人学会更加小心谨慎地处理人际关系，同时也可能对如何表达和接受爱的方式产生一些误解。

这种环境下的经历虽然挑战重重，但也为个人的成长提供了独特的机遇。它教会了您如何在复杂的人际关系中保持自我，学会了在面对冲突时如何保护自己，同时也学会了在适当的时候给予帮助和支持。这些经历虽然不容易，但也是塑造您成为现在这个坚强、独立且有同理心的人的重要因素。

面对这样的家庭背景，您可以尝试以下几个方面来促进自己的心理健康：

1. **寻求支持**：与信任的朋友或家人分享您的感受，或者寻找心理咨询师进行交流，这有助于释放内心的压抑情绪。
2. **情感教育**：通过阅读相关书籍或参加工作坊等方式学习更多关于情感表达的知识，了解如何更健康地处理情绪。
2. **建立界限**：明确自己的情感需求，并勇敢地说出“不”，为自己创造一个更加积极健康的环境。
3. **培养正面的关系**：寻找那些能够给予您支持、理解和爱的人建立联系，这有助于弥补早期缺失的情感体验。
4. **自我反思**：通过日记写作或其他形式的自我反思，探索自己的情感世界，理解自己的内心需求。

记住，每个人的成长路径都是独一无二的，您正在做的每一步都是向着更加健康、幸福的生活迈进的重要步骤。希望您能够继续以开放的心态面对生活中的挑战，找到属于自己的幸福之路。<|im_end|>


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer


user_query = """我生长在缺爱的家族里。父亲的兄弟姐妹间比较寡淡。也会互帮互助，但是总感觉没那么舒心和睦信任。隔着什么似的不亲。在只讲理不讲爱的家族里，大家战战兢兢如履薄冰慌乱，因为喜欢斗来斗去，凡事都要争个对错，大部分时间比较自私有时也会无私（自我保护?）"""

messages = [
    {"role": "system", "content": "你是一位资深的心理咨询师。请基于对话历史，用温暖、专业、共情的语气回复用户。"},
    {"role": "user", "content": user_query}
]



inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=1024)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))


听到你的描述，我能感受到你在这样的家庭环境中所经历的情感压力和孤独感。在一个缺乏亲密情感交流的家庭中成长，确实可能会让人感到与家人之间存在着无形的距离，这种感受是十分真实的。

在这样的家庭氛围下，每个人为了保护自己，可能会采取一些自我保护的方式，比如变得比较自私或者时刻保持警惕。这并不是说你们自私或冷漠，而是因为在这样的环境中，人们学会了如何更好地保护自己，以免受到伤害。但这也意味着，你们可能错过了很多建立深层次连接的机会。

不过，请不要对自己过于苛责。每个人的成长环境都不同，你已经能够意识到这一点，并且愿意寻求帮助，这本身就是一种进步。你可以尝试着慢慢改变这种状态，比如可以先从培养一个亲密的朋友开始，逐渐扩大自己的社交圈，找到那些真正理解和支持你的人。同时，也可以通过阅读书籍、参加工作坊等方式学习更多关于情感沟通的知识，提高自己的情感表达能力。这些都有助于你建立起更加健康和谐的人际关系。

记住，改变是一个逐步的过程，不必急于求成。每一步小小的进步都是值得庆祝的。希望你能逐渐走出阴霾，找到属于自己的幸福和满足感。<|im_end|>


<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support `DPOTrainer` and `GRPOTrainer` for reinforcement learning!

We use our new `UnslothVisionDataCollator` which will help in our vision finetuning setup.

In [ ]:
def formatting_func(examples):
    texts = []
    if isinstance(examples['messages'][0], list):
        for messages in examples['messages']:
            parts = [f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>"
                     for msg in messages]
            texts.append("\n".join(parts))
    else:
        parts = [f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>"
                 for msg in examples['messages']]
        texts.append("\n".join(parts))
    return texts

In [ ]:
train_data

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_data,#.select(range(2000))
    formatting_func = formatting_func,      # 必须指定
    max_seq_length = 1024,
    packing = False,
    args = TrainingArguments(
        # ===== Batch Size（对齐论文 global batch = 80）=====
        per_device_train_batch_size = 8,        # A100 40GB 可开大
        gradient_accumulation_steps = 2,       # 有效 batch = 8 × 10 = 80

        # ===== 训练步数（对齐论文 30,000 steps）=====
        #max_steps = 1000,
        num_train_epochs = 1,

        # ===== 学习率 & Warmup（对齐论文）=====
        learning_rate = 2e-4,
        warmup_steps = 1000,                    # 对齐论文

        # ===== 优化器 & 调度器 =====
        lr_scheduler_type = "cosine",           # cosine 近似论文的 WarmupDecayLR
        optim = "adamw_8bit",
        weight_decay = 0.01,

        # ===== 精度 =====
        bf16 = True,                            # A100 支持 bf16
        fp16 = False,

        # ===== 日志 & 保存 =====
        logging_steps = 20,
        save_steps = 1000,

        # ===== 其他 =====
        seed = 3407,
        output_dir = "/content/outputs",
        report_to = "none",
    ),
)

In [ ]:
# 确保在定义 trainer 之后有这行
trainer.train()

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA L4. Max memory = 22.034 GB.
1.68 GB of memory reserved.


In [ ]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 19,433 | Num Epochs = 3 | Total steps = 912
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 16 x 1) = 64
 "-____-"     Trainable parameters = 21,645,312 of 874,631,232 (2.47% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,2.247522
20,1.826677
30,1.305550
40,1.048336
50,0.972762
60,0.934725
70,0.912254
80,0.899563
90,0.892390
100,0.876668


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

8118.0144 seconds used for training.
135.3 minutes used for training.
Peak reserved memory = 6.293 GB.
Peak reserved memory for training = 4.613 GB.
Peak reserved memory % of max memory = 28.56 %.
Peak reserved memory for training % of max memory = 20.936 %.


<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!

We use `min_p = 0.1` and `temperature = 1.5`. Read this [Tweet](https://x.com/menhguin/status/1826132708508213629) for more information on why.

In [ ]:
import unsloth
from unsloth import FastVisionModel # FastLanguageModel for LLMs
import torch

FastVisionModel.for_inference(model) # Enable for inference!


instruction = """
You are an expert in clinical psychology and conversation analysis. Your task is to analyze a dialogue between a user and an assistant, and produce a structured profile of the user that will guide empathetic response generation.\n\nYou must output exactly four sections, enclosed in the specified XML-like tags. Do not add any text outside these tags.\n\nThe four required sections are:\n1. <ToM> ... </ToM> : A Theory of Mind analysis describing the user's current emotional state, underlying needs, and cognitive perspective, informed by the conversation history.\n2. <Strategy> ... </Strategy> : A concise recommendation of the empathetic strategy to adopt in the next response (e.g., validation, normalization, reflective listening, open-ended question).\n3. <Style> ... </Style> : A description of the user's observed language style preferences (e.g., formality level, vocabulary, tone).\n4. <Context> ... </Context> : Key situational or factual details mentioned by the user that should be remembered for personalized responses.\n
"""

user_query = """
Here is the conversation history:\n\n[User]: Someone was trying to break into my door one night. I had to sit with my kids in their bedroom, calling 911 while my husband guarded the door with a knife and pepper spray. [Assistant]: What?that is so scary/ What happened after that? [User]: Well, the police showed up but the guy fled. Our door was all scratched up and looked like he had been kicking it too. It makes me anxious thinking whoever it is could be anywhere in our small town. [Assistant]: Sorry about that,you might think of moving to somewhere more safe.\n\nGenerate the four sections based on this conversation.
"""

messages = [
    {"role": "system", "content": [
        {"type": "text", "text": instruction}
    ]},
    {'role': 'user',
     'content': user_query},
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    text=input_text, # Explicitly pass as text
    images=None, # No image input for this case
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

#from transformers import TextStreamer
#text_streamer = TextStreamer(tokenizer, skip_prompt = True)
#response = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 1024,use_cache = True, temperature = 1.5, min_p = 0.1)
response = model.generate(**inputs, max_new_tokens = 1024, use_cache = True, temperature = 1.5, min_p = 0.1)
full_response = tokenizer.batch_decode(response, skip_special_tokens=True)[0]
final_text = full_response.split("assistant\n")[-1]
final_text

'<think>\n\n</think>\n\n<ToM>\nThe user appears shaken, anxious, and still somewhat activated by a recent home security breach. They likely feel a mix of fear, vulnerability, and lingering hypervigilance while thinking about possible threats in their small town. Underneath the immediate concern is a need for reassurance, emotional containment, and a sense of safety and control. Their perspective suggests heightened alertness and a strong focus on threat detection, with some lingering worry about the possibility of someone being nearby or in the neighborhood. They may also be sensitive to responses that feel dismissive, minimizing, or overly solution-focused without first acknowledging the seriousness of what happened.\n</ToM>\n\n<Strategy>\nUse calm validation and reflective listening first. Acknowledge the fear and stress, avoid minimizing, and gently invite more detail only if appropriate. If responding, prioritize reassurance, safety-oriented empathy, and a practical next-step quest

# Inference on my test dataset

In [ ]:
import json

def load_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    return data

dataset_test = load_jsonl("/content/qwen35_08b_human_validation.jsonl")
print(len(dataset_test))
print(dataset_test[0])

100
{'custom_id': 'request-213', 'messages': [{'role': 'system', 'content': "You are an expert in clinical psychology and conversation analysis. Your task is to analyze a dialogue between a user and an assistant, and produce a structured profile of the user that will guide empathetic response generation.\n\nYou must output exactly four sections, enclosed in the specified XML-like tags. Do not add any text outside these tags.\n\nThe four required sections are:\n1. <ToM> ... </ToM> : A Theory of Mind analysis describing the user's current emotional state, underlying needs, and cognitive perspective, informed by the conversation history.\n2. <Strategy> ... </Strategy> : A concise recommendation of the empathetic strategy to adopt in the next response (e.g., validation, normalization, reflective listening, open-ended question).\n3. <Style> ... </Style> : A description of the user's observed language style preferences (e.g., formality level, vocabulary, tone).\n4. <Context> ... </Context> :

In [ ]:
dataset_test[1]['messages'][1]['content']

'Here is the conversation history:\n\n[User]: Someone was trying to break into my door one night. I had to sit with my kids in their bedroom, calling 911 while my husband guarded the door with a knife and pepper spray. [Assistant]: What?that is so scary/ What happened after that? [User]: Well, the police showed up but the guy fled. Our door was all scratched up and looked like he had been kicking it too. It makes me anxious thinking whoever it is could be anywhere in our small town. [Assistant]: Sorry about that,you might think of moving to somewhere more safe.\n\nGenerate the four sections based on this conversation.'

In [ ]:
response = model.generate(**inputs, max_new_tokens = 1024, use_cache = True, temperature = 1.5, min_p = 0.1)

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("qwen_lora")  # Local saving
tokenizer.save_pretrained("qwen_lora")
# model.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN") # Online saving

['qwen_lora/processor_config.json']

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastVisionModel
    model, tokenizer = FastVisionModel.from_pretrained(
        model_name = "qwen_lora", # YOUR MODEL YOU USED FOR TRAINING
        load_in_4bit = True, # Set to False for 16bit LoRA
    )
    FastVisionModel.for_inference(model) # Enable for inference!

image = dataset[0]["image"]
instruction = "Write the LaTeX representation for this image."

messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": instruction}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

\frac { N } { M } \in { \bf Z } , \frac { M } { P } \in { \bf Z } , \frac { P } { Q } \in { \bf Z }<|im_end|>


### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens. See [our docs](https://unsloth.ai/docs/basics/inference-and-deployment) for more deployment options.

In [ ]:
# Select ONLY 1 to save! (Both not needed!)

# Save locally to 16bit
if False: model.save_pretrained_merged("unsloth_finetune", tokenizer,)

# To export and save to your Hugging Face account
model.push_to_hub_merged("Spiderman01/finetuned_Qwen_35_08B_empathy", tokenizer, token = "YOUR_HF_TOKEN")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...8B_empathy/tokenizer.json: 100%|##########| 20.0MB / 20.0MB            

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0001-of-00001.safetensors:   5%|5         | 90.7MB / 1.75GB            


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:39<00:00, 39.44s/it]


Unsloth: Merge process complete. Saved to `/content/Spiderman01/finetuned_Qwen_35_08B_empathy`


### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("qwen_finetune", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, token = "YOUR_HF_TOKEN")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method = "f16", token = "YOUR_HF_TOKEN")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("HF_USERNAME/qwen_finetune", tokenizer, quantization_method = "q4_k_m", token = "YOUR_HF_TOKEN")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "HF_USERNAME/qwen_finetune", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "YOUR_HF_TOKEN",
    )

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Looking to use Unsloth locally? Read our [Installation Guide](https://unsloth.ai/docs/get-started/install) for details on installing Unsloth on Windows, Docker, AMD, Intel GPUs.
2. Learn how to do Reinforcement Learning with our [RL Guide and notebooks](https://unsloth.ai/docs/get-started/reinforcement-learning-rl-guide).
3. Read our guides and notebooks for [Text-to-speech (TTS)](https://unsloth.ai/docs/basics/text-to-speech-tts-fine-tuning) and [vision](https://unsloth.ai/docs/basics/vision-fine-tuning) model support.
4. Explore our [LLM Tutorials Directory](https://unsloth.ai/docs/models/tutorials-how-to-fine-tune-and-run-llms) to find dedicated guides for each model.
5. Need help with Inference? Read our [Inference & Deployment page](https://unsloth.ai/docs/basics/inference-and-deployment) for details on using vLLM, llama.cpp, Ollama etc.

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️

  <b>This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)</b>
</div>